# Generate a .dat file for demand peaks

In [1]:
from shared.models import energyscope_original_snapshot_2023
from shared.utils import run_model, load_snapshot

# Generate  2 demand peaks - 2023

You can define several demand peaks by filling the list of dictionaries below. Each dictionary corresponds to a peak and should contain the following information:
- `eud_type`: the type of end-use demand that is peaking, either "heat" for heating demand or "cool" for cooling demand.
- `n_hours`: the number of hours during which the peak occurs.
- `final_demand`: the final demand for domestic heat or cool during the peak in GW or %.
- `final_demand_unit`: the unit of the final heat or cool demand, either "power" for GW (i.e., the domestic heat or cool power demand during the peak), "add_power" for additional GW (i.e., additional power demand expressed in GW during the peak with respect to the monthly average) or "percentage" for % (i.e., the additional power demand in % during the peak with respect to the monthly average).
- `month`: the month of the year during which the peak occurs (1 for January, 2 for February, ..., 12 for December).
- `solar`: a boolean indicating whether the peak occurs during a sunny period (True) or not (False).
- `wind`: a boolean indicating whether the peak occurs during a windy period (True) or not (False).

In [2]:
# Describe 2 peaks
peaks_2_2023 = [
    {#Peak janvier
        "eud_type": "heat", # heat or cool
        "n_hours": 1, # hours during which the peak occurs
        "final_demand": 12.21, # GW or %
        "final_demand_unit": "add_power", # can be "power", "add_power" or "percentage"
        "month": 1, # month of the year (1-12)
        "solar": False, # whether the peak occurs during a sunny period or not
        "wind": False, # whether the peak occurs during a windy period or not
    },
    {# Peak février
        "eud_type": "heat", # heat or cool
        "n_hours": 2, # hours during which the peak occurs
        "final_demand": 21.56, # GW or %
        "final_demand_unit": "add_power", # can be "power", "add_power" or "percentage"
        "month": 2, # month of the year (1-12)
        "solar": False, # whether the peak occurs during a sunny period or not
        "wind": False, # whether the peak occurs during a windy period or not
    }
]

In [6]:
# Load the data from the model
from energyscope.models import Model
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (Avec HP, 2 Pointes, et Séparation Imports - 2023)
energyscope_original_snapshot_2023_updates = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_validation_2023.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    #('mod', str(_PROJECT_DIR / 'imports_split_2023.mod')),       # Nouvelle contrainte spécifique (ex: Churchill Falls)

    # --- DONNÉES SPÉCIFIQUES SUPPLÉMENTAIRES (.dat) ---
    ('mod', str(_PROJECT_DIR / 'imports_split_2023.dat')),       # Nouveaux sets et paramètres de désagrégation
])

results_2023 = run_model(energyscope_original_snapshot_2023_updates,apply_postprocessing=True)
total_heat_demand = float(results_2023.parameters['end_uses_input'].loc['HEAT_LOW_T_SH']['end_uses_input'])  # low temperature heat for space heating
total_cool_demand = float(results_2023.parameters['end_uses_input'].loc['HEAT_LOW_T_SC']['end_uses_input'])  # low temperature heat for space cooling

Gurobi 12.0.3: 

In [16]:
# Write .dat with peak (Version finale sans bug de post-processing)
year, file_name, peaks = 2023, 'peaks_2_2023', peaks_2_2023
n_peaks, out_path = len(peaks), f'../02_Model/{file_name}.dat'
updated_monthly_values = {k: {} for k in ['t_op', 'lighting_month', 'cooling_month', 'heating_month', 'elec_export_month']}

with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    f.write("# This is the data file for demand peaks\n")

    # 1. Extensions des ensembles nécessaires pour AMPL
    new_periods_str = ", ".join([str(13 + i) for i in range(n_peaks)])
    f.write(f"let PERIODS := PERIODS union {{{new_periods_str}}} ;\n")
    f.write(f"let RESOURCES := RESOURCES union {{'ELECTRICITY_OTHER', 'CHURCHILL_FALLS'}} ;\n")

    # =========================================================================
    # CORRECTION POST-PROCESSING : On remplit PEAK_PERIODS pour éviter le RuntimeError
    # =========================================================================
    f.write(f"let PEAK_PERIODS := {{{new_periods_str}}} ;\n\n")
    # =========================================================================

    for i in range(n_peaks):
        eud_type, month, n_hours_peak = peaks[i]['eud_type'], peaks[i]['month'], peaks[i]['n_hours']
        final_demand, final_demand_unit = peaks[i]['final_demand'], peaks[i]['final_demand_unit']

        n_hours_original_period = updated_monthly_values['t_op'].get(month, int(results_2023.parameters['t_op'].loc[month]['t_op']))
        lighting_month = updated_monthly_values['lighting_month'].get(month, float(results_2023.parameters['lighting_month'].loc[month]['lighting_month']))
        cooling_month = updated_monthly_values['cooling_month'].get(month, float(results_2023.parameters['cooling_month'].loc[month]['cooling_month']))
        heating_month = updated_monthly_values['heating_month'].get(month, float(results_2023.parameters['heating_month'].loc[month]['heating_month']))
        elec_export_month = updated_monthly_values['elec_export_month'].get(month, float(results_2023.parameters['elec_export_month'].loc[month]['elec_export_month']))

        if final_demand_unit == "power":
            energy_demand = final_demand * n_hours_peak
        elif final_demand_unit in ["add_power", "percentage"]:
            base_demand = heating_month * total_heat_demand if eud_type == "heat" else cooling_month * total_cool_demand
            if final_demand_unit == "add_power":
                energy_demand = (base_demand / n_hours_original_period + final_demand) * n_hours_peak
            else:
                energy_demand = (base_demand / n_hours_original_period) * (1 + final_demand / 100) * n_hours_peak
        else:
            raise ValueError(f"Invalid final_demand_unit: {final_demand_unit}")

        f.write(f"\n# Peak {i+1}\n")
        f.write(f"let c_op['ELECTRICITY_EHV',{13+i}] := {float(results_2023.parameters['c_op'].loc['ELECTRICITY_EHV', month]['c_op'])} ;\n")
        f.write(f"let c_op['ELECTRICITY_OTHER',{13+i}] := {float(results_2023.parameters['c_op'].loc['ELECTRICITY_OTHER', month]['c_op'])} ;\n")
        f.write(f"let c_op['CHURCHILL_FALLS',{13+i}] := {float(results_2023.parameters['c_op'].loc['CHURCHILL_FALLS', month]['c_op'])} ;\n")

        f.write(f"let t_op[{month}] := {n_hours_original_period - n_hours_peak} ;\nlet t_op[{13+i}] := {n_hours_peak} ;\n")
        updated_monthly_values['t_op'][month] = n_hours_original_period - n_hours_peak

        f.write(f"let lighting_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month} ;\n")
        f.write(f"let lighting_month[{13+i}] := {n_hours_peak / n_hours_original_period * lighting_month} ;\n")
        updated_monthly_values['lighting_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month

        if eud_type == "cool":
            f.write(f"let cooling_month[{month}] := {cooling_month - energy_demand / total_cool_demand} ;\nlet cooling_month[{13+i}] := {energy_demand / total_cool_demand} ;\n")
            updated_monthly_values['cooling_month'][month] = cooling_month - energy_demand / total_cool_demand
        else:
            f.write(f"let cooling_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month} ;\nlet cooling_month[{13+i}] := {n_hours_peak / n_hours_original_period * cooling_month} ;\n")
            updated_monthly_values['cooling_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month

        if eud_type == 'heat':
            f.write(f"let heating_month[{month}] := {heating_month - energy_demand / total_heat_demand} ;\nlet heating_month[{13+i}] := {energy_demand / total_heat_demand} ;\n")
            updated_monthly_values['heating_month'][month] = heating_month - energy_demand / total_heat_demand
        else:
            f.write(f"let heating_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month} ;\nlet heating_month[{13+i}] := {n_hours_peak / n_hours_original_period * heating_month} ;\n")
            updated_monthly_values['heating_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month

        f.write(f"let elec_export_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month} ;\nlet elec_export_month[{13+i}] := {n_hours_peak / n_hours_original_period * elec_export_month} ;\n")
        updated_monthly_values['elec_export_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month

        for tech in ['HYDRO_DAM', 'HYDRO_RIVER', 'NEW_HYDRO_DAM', 'NEW_HYDRO_RIVER']:
            f.write(f"let c_p_t['{tech}',{13+i}] := {float(results_2023.parameters['c_p_t'].loc[tech, month]['c_p_t'])} ;\n")

        w_val = float(results_2023.parameters['c_p_t'].loc['WIND_ONSHORE', month]['c_p_t']) if peaks[i]['wind'] else 0
        f.write(f"let c_p_t['WIND_ONSHORE',{13+i}] := {w_val} ;\nlet c_p_t['WIND_OFFSHORE',{13+i}] := {w_val} ;\n")

        for tech in ['PV_ROOF', 'PV_GROUND', 'DEC_SOLAR']:
            s_val = float(results_2023.parameters['c_p_t'].loc[tech, month]['c_p_t']) if peaks[i]['solar'] else 0
            f.write(f"let c_p_t['{tech}',{13+i}] := {s_val} ;\n")

        f.write(f"let c_p_t['CCGT',{13+i}] := 1 ;\n")

    f.write(f"\nlet c_p['CCGT'] := 1 ;\n")
    for period in range(1, 13):
        f.write(f"let c_p_t['CCGT',{period}] := {float(results_2023.parameters['c_p'].loc['CCGT']['c_p'])} ;\n")

# Generate 108 demand peaks - 2023

In [17]:
#Janvier
# Liste de tes 62 valeurs de demande finale
data_peaks_62_janvier_2023 =  [
    12.21, 10.897, 10.548, 10.425, 10.421, 10.246, 10.103, 9.948,
    9.745, 9.726, 9.713, 9.565, 9.540, 9.375, 9.264, 9.253,
    9.198, 9.153, 9.137, 8.98, 8.892, 8.883, 8.845, 8.726,
    8.611, 8.524, 8.402, 8.261, 8.235, 8.122, 8.09, 7.897,
    7.8, 7.793, 7.755, 7.705, 7.668, 7.553, 7.512, 7.429,
    7.409, 7.337, 7.322, 7.318, 7.23, 7.214, 7.136, 7.059,
    7.051, 6.904, 6.76, 6.76, 6.736, 6.685, 6.586, 6.552,
    6.545, 6.532, 6.523, 6.395, 6.387, 6.36
]

# Génération de la liste des 62 pointes
peaks_62_janvier_2023 = [
    {
        "eud_type": "heat",
        "n_hours": 1,
        "final_demand": value,
        "final_demand_unit": "add_power",
        "month": 1,
        "solar": False,
        "wind": False,
    }
    for value in data_peaks_62_janvier_2023
]

#Février
# Liste des valeurs de demande finale pour le mois 2
data_peaks_56_fevrier_2023 =  [
    21.526, 21.161, 20.403, 19.306, 18.379, 18.271, 17.637, 17.257,
    17.173, 17.121, 17.006, 16.954, 16.892, 16.421, 16.398, 16.267,
    16.042, 16.023, 15.7, 15.379, 15.287, 15.023, 14.891, 14.859,
    14.201, 13.886, 13.841, 13.703, 13.689, 13.552, 13.285, 13.124,
    12.961, 12.874, 12.008, 11.936, 11.933, 11.854, 11.818, 11.662,
    11.518, 11.453, 10.743, 10.533, 10.470, 10.24, 10.16, 9.892,
    9.773, 9.772, 9.632, 9.568, 9.465, 9.139, 9.12, 9.089
]

# Génération de la liste des pointes pour le mois 2
peaks_56_fevrier_2023 = [
    {
        "eud_type": "heat",
        "n_hours": 1,
        "final_demand": value,
        "final_demand_unit": "add_power",
        "month": 2,  # Changement pour le mois 2
        "solar": False,
        "wind": False,
    }
    for value in data_peaks_56_fevrier_2023
]

# Fusionner les deux listes de pointes
peaks_108_2023 = peaks_62_janvier_2023 + peaks_56_fevrier_2023

In [18]:
# Load the data from the model
total_heat_demand = float(results_2023.parameters['end_uses_input'].loc['HEAT_LOW_T_SH']['end_uses_input'])  # low temperature heat for space heating
total_cool_demand = float(results_2023.parameters['end_uses_input'].loc['HEAT_LOW_T_SC']['end_uses_input'])  # low temperature heat for space cooling

In [19]:
# Write .dat with peak (Version finale - 108 pointes)
year, file_name, peaks = 2023, 'peaks_108_2023', peaks_108_2023
n_peaks, out_path = len(peaks), f'../02_Model/{file_name}.dat'
updated_monthly_values = {k: {} for k in ['t_op', 'lighting_month', 'cooling_month', 'heating_month', 'elec_export_month']}

with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    f.write("# This is the data file for demand peaks\n")

    # Extensions et initialisations des ensembles pour AMPL et le post-processing
    new_periods_str = ", ".join([str(13 + i) for i in range(n_peaks)])
    f.write(f"let PERIODS := PERIODS union {{{new_periods_str}}} ;\n")
    f.write(f"let RESOURCES := RESOURCES union {{'ELECTRICITY_OTHER', 'CHURCHILL_FALLS'}} ;\n")
    f.write(f"let PEAK_PERIODS := {{{new_periods_str}}} ;\n\n")

    for i in range(n_peaks):
        # Loading data
        eud_type, month, n_hours_peak = peaks[i]['eud_type'], peaks[i]['month'], peaks[i]['n_hours']
        final_demand, final_demand_unit = peaks[i]['final_demand'], peaks[i]['final_demand_unit']

        n_hours_original_period = updated_monthly_values['t_op'].get(month, int(results_2023.parameters['t_op'].loc[month]['t_op']))
        lighting_month = updated_monthly_values['lighting_month'].get(month, float(results_2023.parameters['lighting_month'].loc[month]['lighting_month']))
        cooling_month = updated_monthly_values['cooling_month'].get(month, float(results_2023.parameters['cooling_month'].loc[month]['cooling_month']))
        heating_month = updated_monthly_values['heating_month'].get(month, float(results_2023.parameters['heating_month'].loc[month]['heating_month']))
        elec_export_month = updated_monthly_values['elec_export_month'].get(month, float(results_2023.parameters['elec_export_month'].loc[month]['elec_export_month']))

        if final_demand_unit == "power":
            energy_demand = final_demand * n_hours_peak
        elif final_demand_unit in ["add_power", "percentage"]:
            base_demand = heating_month * total_heat_demand if eud_type == "heat" else cooling_month * total_cool_demand
            if final_demand_unit == "add_power":
                energy_demand = (base_demand / n_hours_original_period + final_demand) * n_hours_peak
            else:
                energy_demand = (base_demand / n_hours_original_period) * (1 + final_demand / 100) * n_hours_peak
        else:
            raise ValueError(f"Invalid final_demand_unit: {final_demand_unit}")

        # Write data in the .dat file
        f.write(f"\n# Peak {i+1}\n")
        f.write(f"let c_op['ELECTRICITY_EHV',{13+i}] := {float(results_2023.parameters['c_op'].loc['ELECTRICITY_EHV', month]['c_op'])} ;\n")
        f.write(f"let c_op['ELECTRICITY_OTHER',{13+i}] := {float(results_2023.parameters['c_op'].loc['ELECTRICITY_OTHER', month]['c_op'])} ;\n")
        f.write(f"let c_op['CHURCHILL_FALLS',{13+i}] := {float(results_2023.parameters['c_op'].loc['CHURCHILL_FALLS', month]['c_op'])} ;\n")

        # Period length and distribution
        f.write(f"let t_op[{month}] := {n_hours_original_period - n_hours_peak} ;\nlet t_op[{13+i}] := {n_hours_peak} ;\n")
        updated_monthly_values['t_op'][month] = n_hours_original_period - n_hours_peak

        f.write(f"let lighting_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month} ;\n")
        f.write(f"let lighting_month[{13+i}] := {n_hours_peak / n_hours_original_period * lighting_month} ;\n")
        updated_monthly_values['lighting_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month

        # Cooling / Heating / Export
        if eud_type == "cool":
            f.write(f"let cooling_month[{month}] := {cooling_month - energy_demand / total_cool_demand} ;\nlet cooling_month[{13+i}] := {energy_demand / total_cool_demand} ;\n")
            updated_monthly_values['cooling_month'][month] = cooling_month - energy_demand / total_cool_demand
        else:
            f.write(f"let cooling_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month} ;\nlet cooling_month[{13+i}] := {n_hours_peak / n_hours_original_period * cooling_month} ;\n")
            updated_monthly_values['cooling_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month

        if eud_type == 'heat':
            f.write(f"let heating_month[{month}] := {heating_month - energy_demand / total_heat_demand} ;\nlet heating_month[{13+i}] := {energy_demand / total_heat_demand} ;\n")
            updated_monthly_values['heating_month'][month] = heating_month - energy_demand / total_heat_demand
        else:
            f.write(f"let heating_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month} ;\nlet heating_month[{13+i}] := {n_hours_peak / n_hours_original_period * heating_month} ;\n")
            updated_monthly_values['heating_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month

        f.write(f"let elec_export_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month} ;\nlet elec_export_month[{13+i}] := {n_hours_peak / n_hours_original_period * elec_export_month} ;\n")
        updated_monthly_values['elec_export_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month

        # Renewables & CCGTs Capacity Factors
        for tech in ['HYDRO_DAM', 'HYDRO_RIVER', 'NEW_HYDRO_DAM', 'NEW_HYDRO_RIVER']:
            f.write(f"let c_p_t['{tech}',{13+i}] := {float(results_2023.parameters['c_p_t'].loc[tech, month]['c_p_t'])} ;\n")

        w_val = float(results_2023.parameters['c_p_t'].loc['WIND_ONSHORE', month]['c_p_t']) if peaks[i]['wind'] else 0
        f.write(f"let c_p_t['WIND_ONSHORE',{13+i}] := {w_val} ;\nlet c_p_t['WIND_OFFSHORE',{13+i}] := {w_val} ;\n")

        for tech in ['PV_ROOF', 'PV_GROUND', 'DEC_SOLAR']:
            s_val = float(results_2023.parameters['c_p_t'].loc[tech, month]['c_p_t']) if peaks[i]['solar'] else 0
            f.write(f"let c_p_t['{tech}',{13+i}] := {s_val} ;\n")

        f.write(f"let c_p_t['CCGT',{13+i}] := 1 ;\n")

    f.write(f"\nlet c_p['CCGT'] := 1 ;\n")
    for period in range(1, 13):
        f.write(f"let c_p_t['CCGT',{period}] := {float(results_2023.parameters['c_p'].loc['CCGT']['c_p'])} ;\n")

# Generate 2 demande peaks - 2050

In [21]:
# Describe 2 peaks
peaks_2_2050 = [
    {#Peak janvier
        "eud_type": "heat", # heat or cool
        "n_hours": 1, # hours during which the peak occurs
        "final_demand": 15, # GW or %
        "final_demand_unit": "add_power", # can be "power", "add_power" or "percentage"
        "month": 1, # month of the year (1-12)
        "solar": False, # whether the peak occurs during a sunny period or not
        "wind": False, # whether the peak occurs during a windy period or not
    },
    {# Peak février
        "eud_type": "heat", # heat or cool
        "n_hours": 2, # hours during which the peak occurs
        "final_demand": 15, # GW or %
        "final_demand_unit": "add_power", # can be "power", "add_power" or "percentage"
        "month": 2, # month of the year (1-12)
        "solar": False, # whether the peak occurs during a sunny period or not
        "wind": False, # whether the peak occurs during a windy period or not
    }
]

In [22]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_original_snapshot_2050_update = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),     # Déclarations / structures pour les HP
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),    # Logique hivernale des HP
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro

    # --- DONNÉES ET SCRIPTS DE CONTEXTE ---
    # 2. Flag 'mod' pour basculer AMPL en mode script juste avant les données HP
    ('mod', str(_PROJECT_DIR / 'HP_extra.dat')),     # Ton fichier (vide ou avec commentaires) qui ouvre les droits du 'let'

    # 3. Injection des données des pompes à chaleur
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),     # Profite du mode script pour exécuter son 'let cop_normal'
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])
#Load the data from the model
results_2050 = run_model(energyscope_original_snapshot_2050_update,apply_postprocessing=True)
total_heat_demand = float(results_2050.parameters['end_uses_input'].loc['HEAT_LOW_T_SH']['end_uses_input'])  # low temperature heat for space heating
total_cool_demand = float(results_2050.parameters['end_uses_input'].loc['HEAT_LOW_T_SC']['end_uses_input'])  # low temperature heat for space cooling

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [23]:
# Write .dat with peak (Version finale - 2050)
year, file_name, peaks = 2050, 'peaks_2_2050', peaks_2_2050
n_peaks, out_path = len(peaks), f'../02_Model/{file_name}.dat'
updated_monthly_values = {k: {} for k in ['t_op', 'lighting_month', 'cooling_month', 'heating_month', 'elec_export_month']}

with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    f.write("# This is the data file for demand peaks\n")

    # Extensions et initialisations des ensembles pour AMPL et le post-processing
    new_periods_str = ", ".join([str(13 + i) for i in range(n_peaks)])
    f.write(f"let PERIODS := PERIODS union {{{new_periods_str}}} ;\n")
    f.write(f"let RESOURCES := RESOURCES union {{'ELECTRICITY_OTHER', 'CHURCHILL_FALLS'}} ;\n")
    f.write(f"let PEAK_PERIODS := {{{new_periods_str}}} ;\n\n")

    for i in range(n_peaks):
        # Loading data
        eud_type, month, n_hours_peak = peaks[i]['eud_type'], peaks[i]['month'], peaks[i]['n_hours']
        final_demand, final_demand_unit = peaks[i]['final_demand'], peaks[i]['final_demand_unit']

        n_hours_original_period = updated_monthly_values['t_op'].get(month, int(results_2050.parameters['t_op'].loc[month]['t_op']))
        lighting_month = updated_monthly_values['lighting_month'].get(month, float(results_2050.parameters['lighting_month'].loc[month]['lighting_month']))
        cooling_month = updated_monthly_values['cooling_month'].get(month, float(results_2050.parameters['cooling_month'].loc[month]['cooling_month']))
        heating_month = updated_monthly_values['heating_month'].get(month, float(results_2050.parameters['heating_month'].loc[month]['heating_month']))
        elec_export_month = updated_monthly_values['elec_export_month'].get(month, float(results_2050.parameters['elec_export_month'].loc[month]['elec_export_month']))

        if final_demand_unit == "power":
            energy_demand = final_demand * n_hours_peak
        elif final_demand_unit in ["add_power", "percentage"]:
            base_demand = heating_month * total_heat_demand if eud_type == "heat" else cooling_month * total_cool_demand
            if final_demand_unit == "add_power":
                energy_demand = (base_demand / n_hours_original_period + final_demand) * n_hours_peak
            else:
                energy_demand = (base_demand / n_hours_original_period) * (1 + final_demand / 100) * n_hours_peak
        else:
            raise ValueError(f"Invalid final_demand_unit: {final_demand_unit}")

        # Write data in the .dat file
        f.write(f"\n# Peak {i+1}\n")
        f.write(f"let c_op['ELECTRICITY_EHV',{13+i}] := {float(results_2050.parameters['c_op'].loc['ELECTRICITY_EHV', month]['c_op'])} ;\n")
        f.write(f"let c_op['ELECTRICITY_OTHER',{13+i}] := {float(results_2050.parameters['c_op'].loc['ELECTRICITY_OTHER', month]['c_op'])} ;\n")
        f.write(f"let c_op['CHURCHILL_FALLS',{13+i}] := {float(results_2050.parameters['c_op'].loc['CHURCHILL_FALLS', month]['c_op'])} ;\n")

        # Period length and distribution
        f.write(f"let t_op[{month}] := {n_hours_original_period - n_hours_peak} ;\nlet t_op[{13+i}] := {n_hours_peak} ;\n")
        updated_monthly_values['t_op'][month] = n_hours_original_period - n_hours_peak

        f.write(f"let lighting_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month} ;\n")
        f.write(f"let lighting_month[{13+i}] := {n_hours_peak / n_hours_original_period * lighting_month} ;\n")
        updated_monthly_values['lighting_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * lighting_month

        # Cooling / Heating / Export
        if eud_type == "cool":
            f.write(f"let cooling_month[{month}] := {cooling_month - energy_demand / total_cool_demand} ;\nlet cooling_month[{13+i}] := {energy_demand / total_cool_demand} ;\n")
            updated_monthly_values['cooling_month'][month] = cooling_month - energy_demand / total_cool_demand
        else:
            f.write(f"let cooling_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month} ;\nlet cooling_month[{13+i}] := {n_hours_peak / n_hours_original_period * cooling_month} ;\n")
            updated_monthly_values['cooling_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * cooling_month

        if eud_type == 'heat':
            f.write(f"let heating_month[{month}] := {heating_month - energy_demand / total_heat_demand} ;\nlet heating_month[{13+i}] := {energy_demand / total_heat_demand} ;\n")
            updated_monthly_values['heating_month'][month] = heating_month - energy_demand / total_heat_demand
        else:
            f.write(f"let heating_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month} ;\nlet heating_month[{13+i}] := {n_hours_peak / n_hours_original_period * heating_month} ;\n")
            updated_monthly_values['heating_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * heating_month

        f.write(f"let elec_export_month[{month}] := {(n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month} ;\nlet elec_export_month[{13+i}] := {n_hours_peak / n_hours_original_period * elec_export_month} ;\n")
        updated_monthly_values['elec_export_month'][month] = (n_hours_original_period - n_hours_peak) / n_hours_original_period * elec_export_month

        # Renewables & CCGTs Capacity Factors
        for tech in ['HYDRO_DAM', 'HYDRO_RIVER', 'NEW_HYDRO_DAM', 'NEW_HYDRO_RIVER']:
            f.write(f"let c_p_t['{tech}',{13+i}] := {float(results_2050.parameters['c_p_t'].loc[tech, month]['c_p_t'])} ;\n")

        w_val = float(results_2050.parameters['c_p_t'].loc['WIND_ONSHORE', month]['c_p_t']) if peaks[i]['wind'] else 0
        f.write(f"let c_p_t['WIND_ONSHORE',{13+i}] := {w_val} ;\nlet c_p_t['WIND_OFFSHORE',{13+i}] := {w_val} ;\n")

        for tech in ['PV_ROOF', 'PV_GROUND', 'DEC_SOLAR']:
            s_val = float(results_2050.parameters['c_p_t'].loc[tech, month]['c_p_t']) if peaks[i]['solar'] else 0
            f.write(f"let c_p_t['{tech}',{13+i}] := {s_val} ;\n")

        f.write(f"let c_p_t['CCGT',{13+i}] := 1 ;\n")

    f.write(f"\nlet c_p['CCGT'] := 1 ;\n")
    for period in range(1, 13):
        f.write(f"let c_p_t['CCGT',{period}] := {float(results_2050.parameters['c_p'].loc['CCGT']['c_p'])} ;\n")